# Notebook 08: Spatial correlation model and validation

This notebook implements the first Phase 2 milestone. It builds and validates
the within-event spatial-correlation transforms before any full-catalog ground
motion, damage, or financial outputs are changed.

The experiment contains three cases:

- **I0_PHASE1_INDEPENDENT**: exact Phase 1 control;
- **C1_ALDEA22_SUBDUCTION**: primary interface and intraslab subduction case;
- **C2_GODA_ATKINSON09**: subduction-environment sensitivity case.

The annual catalog, event residuals, marginal ground-motion distributions,
damage uniforms, exposure, insurance terms, and Phase 1 reinsurance layer remain
frozen. Only the within-event cross-site dependence changes.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook8_spatial_correlation_model_validation_v2"
VALIDATION_SEED = 20260828
EMPIRICAL_REALIZATIONS = 4000
EXPECTED_SITE_COUNT = 470
EXPECTED_UNIQUE_COORDINATES = 426
EXPECTED_ZERO_MODES = 44
EXPECTED_SITE_ORDER_CRLF_SHA256 = (
    "fc64d53410fdadab5379f82c73c1d74ce3393911fbdd73a0dbc483f056f3d100"
)
EXPECTED_CROSS_IMT_RHO = 0.7321409900247263


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "08_spatial_correlation_model_and_validation.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_4_cell_18_site_order.csv"
        ).exists() and (candidate / "tools" / "spatial_correlation.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the seismic-correlation-insurance-loss repository root."
    )


def project_relative_path(path: Path) -> str:
    """Return a portable repository-relative path using POSIX separators."""
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError as exc:
        raise ValueError(
            f"Artifact path is outside the project root: {path}"
        ) from exc


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def canonical_crlf_sha256(path: Path) -> str:
    text = path.read_text(encoding="utf-8")
    normalized = "\r\n".join(text.splitlines()) + "\r\n"
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tools.spatial_correlation import (
    ALDEA_ALPHA,
    ALDEA_PGA_BETA_KM,
    ALDEA_SA0P4_BETA_KM,
    CASE_C1,
    CASE_C2,
    CASE_I0,
    DEFAULT_CROSS_IMT_RHO,
    GODA_ALPHA,
    GODA_BETA,
    GODA_GAMMA,
    SUPPORTED_CASES,
    aldea_correlation,
    build_correlation_case,
    equal_weight_effective_site_count,
    geometry_summary,
    goda_atkinson_correlation,
    haversine_distance_matrix,
    mean_off_diagonal_correlation,
    transform_site_latents,
)

SITE_ORDER_PATH = (
    PROJECT_ROOT / "data" / "metadata" / "notebook_4_cell_18_site_order.csv"
)
RANDOM_SPEC_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_4_cell_19_random_stream_specification.json"
)
NOTEBOOK4_HANDOFF_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_4_final_handoff"
    / "notebook_5_input_handoff.json"
)
METADATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "phase_2"
    / "notebook_8_spatial_correlation"
)
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "phase_2"
    / "notebook_8_spatial_correlation"
)
PLOT_DIR = PROCESSED_DIR / "plots"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

site_order = pd.read_csv(SITE_ORDER_PATH)
random_spec = load_json(RANDOM_SPEC_PATH)
notebook4_handoff = load_json(NOTEBOOK4_HANDOFF_PATH)

required_columns = {
    "site_ordinal",
    "site_id",
    "site_longitude",
    "site_latitude",
}
cell1_checks: list[dict[str, Any]] = []
append_check(
    cell1_checks,
    "site_order_columns_complete",
    required_columns.issubset(site_order.columns),
    f"Columns: {sorted(site_order.columns.tolist())}",
)
append_check(
    cell1_checks,
    "site_order_row_count",
    len(site_order) == EXPECTED_SITE_COUNT,
    f"Rows: {len(site_order):,}; expected: {EXPECTED_SITE_COUNT:,}",
)
append_check(
    cell1_checks,
    "site_ordinals_frozen",
    np.array_equal(
        site_order["site_ordinal"].to_numpy(),
        np.arange(EXPECTED_SITE_COUNT),
    ),
    "Required site_ordinal sequence: 0 through 469.",
)
site_order_raw_sha256 = sha256_file(SITE_ORDER_PATH)
site_order_crlf_sha256 = canonical_crlf_sha256(SITE_ORDER_PATH)
append_check(
    cell1_checks,
    "site_order_canonical_hash_frozen",
    site_order_crlf_sha256 == EXPECTED_SITE_ORDER_CRLF_SHA256,
    (
        f"Canonical CRLF SHA-256: {site_order_crlf_sha256}; "
        f"raw checkout SHA-256: {site_order_raw_sha256}."
    ),
)
append_check(
    cell1_checks,
    "phase1_random_generator_frozen",
    random_spec.get("generator") == "numpy.random.PCG64DXSM",
    f"Generator: {random_spec.get('generator')}",
)
append_check(
    cell1_checks,
    "phase1_site_latent_transform_frozen",
    random_spec.get("latent_transform")
    == {
        "pga": "z1",
        "sa0p4": "rho*z1 + sqrt(1-rho^2)*z2",
    },
    f"Transform: {random_spec.get('latent_transform')}",
)
handoff_rho = float(notebook4_handoff["aleatory_model"]["cross_imt_rho"])
append_check(
    cell1_checks,
    "cross_imt_rho_frozen",
    handoff_rho == EXPECTED_CROSS_IMT_RHO == DEFAULT_CROSS_IMT_RHO,
    f"Frozen rho: {handoff_rho:.16f}",
)

cell1_validation = pd.DataFrame(cell1_checks)
cell1_validation_path = METADATA_DIR / "notebook_8_cell_1_validation.csv"
cell1_validation.to_csv(cell1_validation_path, index=False)
if not bool(cell1_validation.loc[
    cell1_validation["severity"].eq("critical"), "passed"
].all()):
    raise RuntimeError("Notebook 08 Cell 1 validation failed.")

write_json(
    METADATA_DIR / "notebook_8_cell_1_summary.json",
    {
        "pipeline_version": PIPELINE_VERSION,
        "project_root": ".",
        "site_count": int(len(site_order)),
        "site_order_raw_sha256": site_order_raw_sha256,
        "site_order_canonical_crlf_sha256": site_order_crlf_sha256,
        "phase1_random_specification": project_relative_path(RANDOM_SPEC_PATH),
        "phase1_random_specification_sha256": sha256_file(RANDOM_SPEC_PATH),
        "notebook4_handoff": project_relative_path(NOTEBOOK4_HANDOFF_PATH),
        "notebook4_handoff_sha256": sha256_file(NOTEBOOK4_HANDOFF_PATH),
        "critical_checks": int(cell1_validation["severity"].eq("critical").sum()),
        "critical_failures": int(
            (
                cell1_validation["severity"].eq("critical")
                & ~cell1_validation["passed"]
            ).sum()
        ),
    },
)

print("=" * 78)
print("NOTEBOOK 08 CELL 1: FROZEN PHASE 1 CONTROLS VALIDATED")
print("=" * 78)
print("Project root:                 .")
print(f"Frozen sites:                {len(site_order):,}")
print(f"Canonical site-order hash:   {site_order_crlf_sha256}")
print(f"Cross-IMT rho:               {handoff_rho:.16f}")
print(f"Critical checks:             {len(cell1_validation):,}")
print("Next: construct the site-distance and correlation matrices.")

## Joint residual construction

The Phase 1 random streams are retained. For the Aldea case,

$$
\epsilon_{\mathrm{PGA}}=C_{\mathrm{PGA}}^{1/2}z_1,
$$

$$
\epsilon_{\mathrm{SA}(0.4)}=
\rho_T\epsilon_{\mathrm{PGA}}+
\sqrt{1-\rho_T^2}\,V^{1/2}z_2,
$$

where

$$
V=\frac{C_{\mathrm{SA}(0.4)}-\rho_T^2C_{\mathrm{PGA}}}
{1-\rho_T^2}.
$$

This preserves both published spatial marginals and the frozen same-site
PGA versus SA(0.4 s) correlation. A symmetric spectral square root is used
because the 44 duplicated coordinates create 44 structural zero modes.
The factor is then projected onto the exact duplicate-row subspaces so
co-located buildings receive numerically identical residuals on every
supported platform.

In [ ]:
longitude = site_order["site_longitude"].to_numpy(dtype=float)
latitude = site_order["site_latitude"].to_numpy(dtype=float)
distance_km = haversine_distance_matrix(longitude, latitude)
portfolio_geometry = geometry_summary(longitude, latitude, distance_km)

cases = {
    case_name: build_correlation_case(
        distance_km,
        case_name=case_name,
        cross_imt_rho=EXPECTED_CROSS_IMT_RHO,
    )
    for case_name in SUPPORTED_CASES
}

diagnostic_rows: list[dict[str, Any]] = []
case_summary_rows: list[dict[str, Any]] = []
for case_name, case in cases.items():
    for diagnostic in case.diagnostics:
        diagnostic_rows.append(
            {"case_name": case_name, **diagnostic.to_dict()}
        )
    for imt_name, matrix in (
        ("PGA", case.pga_correlation),
        ("SA0P4", case.sa0p4_correlation),
    ):
        case_summary_rows.append(
            {
                "case_name": case_name,
                "imt": imt_name,
                "mean_off_diagonal_correlation": (
                    mean_off_diagonal_correlation(matrix)
                ),
                "equal_weight_effective_site_count": (
                    equal_weight_effective_site_count(matrix)
                ),
                "correlation_at_maximum_separation": float(
                    matrix[np.unravel_index(
                        np.argmax(distance_km), distance_km.shape
                    )]
                ),
            }
        )

matrix_diagnostics = pd.DataFrame(diagnostic_rows)
case_summary = pd.DataFrame(case_summary_rows)
matrix_diagnostics_path = METADATA_DIR / "notebook_8_matrix_diagnostics.csv"
case_summary_path = METADATA_DIR / "notebook_8_case_summary.csv"
matrix_diagnostics.to_csv(matrix_diagnostics_path, index=False)
case_summary.to_csv(case_summary_path, index=False)

factor_path = PROCESSED_DIR / "spatial_correlation_factors.npz"
factor_path.parent.mkdir(parents=True, exist_ok=True)
factor_payload: dict[str, np.ndarray] = {"distance_km": distance_km}
case_prefixes = {CASE_I0: "i0", CASE_C1: "c1", CASE_C2: "c2"}
for case_name, case in cases.items():
    prefix = case_prefixes[case_name]
    factor_payload[f"{prefix}_pga_correlation"] = case.pga_correlation
    factor_payload[f"{prefix}_sa0p4_correlation"] = case.sa0p4_correlation
    factor_payload[f"{prefix}_conditional_correlation"] = (
        case.conditional_correlation
    )
    factor_payload[f"{prefix}_pga_square_root"] = case.pga_square_root
    factor_payload[f"{prefix}_conditional_square_root"] = (
        case.conditional_square_root
    )
temporary_factor_path = factor_path.with_suffix(".npz.tmp")
with temporary_factor_path.open("wb") as handle:
    np.savez_compressed(handle, **factor_payload)
temporary_factor_path.replace(factor_path)

model_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "frozen_phase1_controls": {
        "site_order_canonical_crlf_sha256": site_order_crlf_sha256,
        "cross_imt_rho": EXPECTED_CROSS_IMT_RHO,
        "random_generator": random_spec["generator"],
        "latent_transform": random_spec["latent_transform"],
    },
    "geometry": portfolio_geometry,
    "cases": {
        case_name: case.metadata() for case_name, case in cases.items()
    },
    "parameters": {
        "aldea_alpha": ALDEA_ALPHA,
        "aldea_pga_beta_km": ALDEA_PGA_BETA_KM,
        "aldea_sa0p4_beta_km": ALDEA_SA0P4_BETA_KM,
        "goda_alpha": GODA_ALPHA,
        "goda_beta": GODA_BETA,
        "goda_gamma": GODA_GAMMA,
    },
    "factorization": {
        "method": (
            "unique symmetric positive-semidefinite spectral square root "
            "with exact duplicate-row subspace projection"
        ),
        "material_negative_eigenvalue_threshold": -1.0e-10,
        "numerical_rank_threshold": 1.0e-10,
        "factor_path": project_relative_path(factor_path),
        "factor_sha256": sha256_file(factor_path),
    },
    "references": [
        {
            "citation": "Aldea, Heresi, and Pasten (2022)",
            "doi": "10.1002/eqe.3674",
        },
        {
            "citation": "Goda and Atkinson (2009)",
            "doi": "10.1785/0120090007",
        },
        {
            "citation": "Baker and Jayaram (2008)",
            "doi": "10.1193/1.2857544",
        },
    ],
}
model_specification_path = METADATA_DIR / "notebook_8_model_specification.json"
geometry_path = METADATA_DIR / "notebook_8_geometry_summary.json"
write_json(model_specification_path, model_specification)
write_json(geometry_path, portfolio_geometry)

print("=" * 78)
print("NOTEBOOK 08 CELL 2: CORRELATION MATRICES CONSTRUCTED")
print("=" * 78)
print(f"Sites:                       {portfolio_geometry['site_count']:,}")
print(f"Unique coordinates:          {portfolio_geometry['unique_coordinate_count']:,}")
print(f"Structural zero modes:       {EXPECTED_ZERO_MODES:,}")
print(f"Median pair distance:        {portfolio_geometry['median_pair_distance_km']:.4f} km")
print(f"Maximum pair distance:       {portfolio_geometry['maximum_pair_distance_km']:.4f} km")
print(f"Factor artifact:             {project_relative_path(factor_path)}")
display(case_summary)

In [ ]:
analytic_checks: list[dict[str, Any]] = []

append_check(
    analytic_checks,
    "geometry_site_count",
    portfolio_geometry["site_count"] == EXPECTED_SITE_COUNT,
    f"Sites: {portfolio_geometry['site_count']}",
)
append_check(
    analytic_checks,
    "geometry_unique_coordinate_count",
    portfolio_geometry["unique_coordinate_count"]
    == EXPECTED_UNIQUE_COORDINATES,
    f"Unique coordinates: {portfolio_geometry['unique_coordinate_count']}",
)
append_check(
    analytic_checks,
    "all_matrices_positive_semidefinite",
    bool(matrix_diagnostics["material_negative_eigenvalue_count"].eq(0).all()),
    (
        "Minimum eigenvalue across all matrices: "
        f"{matrix_diagnostics['minimum_eigenvalue'].min():.6e}"
    ),
)
append_check(
    analytic_checks,
    "all_factor_reconstructions_accurate",
    bool(
        matrix_diagnostics["reconstruction_error_max_abs"]
        .le(1.0e-12)
        .all()
    ),
    (
        "Maximum reconstruction error: "
        f"{matrix_diagnostics['reconstruction_error_max_abs'].max():.6e}"
    ),
)
append_check(
    analytic_checks,
    "correlated_case_zero_modes_match_duplicate_coordinates",
    bool(
        matrix_diagnostics.loc[
            matrix_diagnostics["case_name"].isin([CASE_C1, CASE_C2]),
            "zero_mode_count",
        ].eq(EXPECTED_ZERO_MODES).all()
    ),
    "All C1 and C2 matrices must have 44 structural zero modes.",
)

identity = np.eye(EXPECTED_SITE_COUNT)
independent = cases[CASE_I0]
append_check(
    analytic_checks,
    "independent_case_identity_matrices",
    bool(
        np.array_equal(independent.pga_correlation, identity)
        and np.array_equal(independent.sa0p4_correlation, identity)
        and np.array_equal(independent.conditional_correlation, identity)
        and np.array_equal(independent.pga_square_root, identity)
        and np.array_equal(independent.conditional_square_root, identity)
    ),
    "I0 matrices and factors reproduce exact identity transforms.",
)

aldea = cases[CASE_C1]
reconstructed_aldea_sa = (
    EXPECTED_CROSS_IMT_RHO**2 * aldea.pga_correlation
    + (1.0 - EXPECTED_CROSS_IMT_RHO**2)
    * aldea.conditional_correlation
)
aldea_sa_error = float(
    np.max(np.abs(reconstructed_aldea_sa - aldea.sa0p4_correlation))
)
append_check(
    analytic_checks,
    "aldea_joint_construction_recovers_sa0p4_covariance",
    aldea_sa_error <= 2.0e-15,
    f"Maximum covariance identity error: {aldea_sa_error:.6e}",
)

goda = cases[CASE_C2]
goda_common_error = float(
    max(
        np.max(np.abs(goda.pga_correlation - goda.sa0p4_correlation)),
        np.max(np.abs(goda.pga_correlation - goda.conditional_correlation)),
    )
)
append_check(
    analytic_checks,
    "goda_common_kernel_preserved",
    goda_common_error <= 2.0e-15,
    f"Maximum common-kernel error: {goda_common_error:.6e}",
)

coordinates = np.column_stack([longitude, latitude])
_, coordinate_inverse, coordinate_counts = np.unique(
    coordinates,
    axis=0,
    return_inverse=True,
    return_counts=True,
)
colocated_groups = [
    np.flatnonzero(coordinate_inverse == group)
    for group in np.flatnonzero(coordinate_counts > 1)
]
validation_rng = np.random.Generator(np.random.PCG64DXSM(VALIDATION_SEED))
validation_z1 = validation_rng.standard_normal(EXPECTED_SITE_COUNT)
validation_z2 = validation_rng.standard_normal(EXPECTED_SITE_COUNT)
maximum_colocated_difference = 0.0
for case_name in (CASE_C1, CASE_C2):
    epsilon_pga, epsilon_sa = transform_site_latents(
        cases[case_name], validation_z1, validation_z2
    )
    for members in colocated_groups:
        maximum_colocated_difference = max(
            maximum_colocated_difference,
            float(np.max(np.abs(epsilon_pga[members] - epsilon_pga[members[0]]))),
            float(np.max(np.abs(epsilon_sa[members] - epsilon_sa[members[0]]))),
        )
append_check(
    analytic_checks,
    "colocated_residuals_identical",
    maximum_colocated_difference <= 1.0e-12,
    f"Maximum co-located residual difference: {maximum_colocated_difference:.6e}",
)

i0_pga, i0_sa = transform_site_latents(
    independent, validation_z1, validation_z2
)
expected_i0_sa = (
    EXPECTED_CROSS_IMT_RHO * validation_z1
    + np.sqrt(1.0 - EXPECTED_CROSS_IMT_RHO**2) * validation_z2
)
append_check(
    analytic_checks,
    "independent_case_phase1_transform_exact",
    bool(
        np.array_equal(i0_pga, validation_z1)
        and np.array_equal(i0_sa, expected_i0_sa)
    ),
    "I0 reproduces the Phase 1 normalized latent transform exactly.",
)

analytic_validation = pd.DataFrame(analytic_checks)
analytic_validation_path = METADATA_DIR / "notebook_8_analytic_validation.csv"
analytic_validation.to_csv(analytic_validation_path, index=False)
if not bool(analytic_validation.loc[
    analytic_validation["severity"].eq("critical"), "passed"
].all()):
    display(analytic_validation)
    raise RuntimeError("Notebook 08 analytic validation failed.")

print("=" * 78)
print("NOTEBOOK 08 CELL 3: ANALYTIC VALIDATION PASSED")
print("=" * 78)
print(f"Critical checks:              {len(analytic_validation):,}")
print(f"Maximum co-location error:    {maximum_colocated_difference:.6e}")
print(f"Aldea covariance error:       {aldea_sa_error:.6e}")
print(f"Goda common-kernel error:     {goda_common_error:.6e}")
display(analytic_validation)

## Empirical Monte Carlo check

The exact matrix checks above are the production acceptance tests. This cell adds
an empirical simulation check using the same raw latent vectors for all three
cases. It verifies marginal standard-normal behavior, selected site-pair
correlations, and the same-site PGA versus SA(0.4 s) correlation.

In [ ]:
upper_i, upper_j = np.triu_indices(EXPECTED_SITE_COUNT, k=1)
upper_distances = distance_km[upper_i, upper_j]
target_probabilities = [0.00, 0.05, 0.25, 0.50, 0.75, 0.95, 1.00]
selected_pairs: list[tuple[int, int]] = []
for probability in target_probabilities:
    target_distance = float(np.quantile(upper_distances, probability))
    candidate = int(np.argmin(np.abs(upper_distances - target_distance)))
    pair = (int(upper_i[candidate]), int(upper_j[candidate]))
    if pair not in selected_pairs:
        selected_pairs.append(pair)

empirical_rng = np.random.Generator(np.random.PCG64DXSM(VALIDATION_SEED))
empirical_z1 = empirical_rng.standard_normal(
    (EXPECTED_SITE_COUNT, EMPIRICAL_REALIZATIONS)
)
empirical_z2 = empirical_rng.standard_normal(
    (EXPECTED_SITE_COUNT, EMPIRICAL_REALIZATIONS)
)

empirical_rows: list[dict[str, Any]] = []
empirical_check_rows: list[dict[str, Any]] = []
for case_name, case in cases.items():
    epsilon_pga, epsilon_sa = transform_site_latents(
        case, empirical_z1, empirical_z2
    )
    pga_mean_by_site = np.mean(epsilon_pga, axis=1)
    sa_mean_by_site = np.mean(epsilon_sa, axis=1)
    pga_std_by_site = np.std(epsilon_pga, axis=1, ddof=1)
    sa_std_by_site = np.std(epsilon_sa, axis=1, ddof=1)
    pga_sa_rho_by_site = np.array(
        [
            np.corrcoef(epsilon_pga[index], epsilon_sa[index])[0, 1]
            for index in range(EXPECTED_SITE_COUNT)
        ]
    )

    maximum_selected_pair_error = 0.0
    for site_i, site_j in selected_pairs:
        empirical_pga_rho = float(
            np.corrcoef(epsilon_pga[site_i], epsilon_pga[site_j])[0, 1]
        )
        empirical_sa_rho = float(
            np.corrcoef(epsilon_sa[site_i], epsilon_sa[site_j])[0, 1]
        )
        target_pga_rho = float(case.pga_correlation[site_i, site_j])
        target_sa_rho = float(case.sa0p4_correlation[site_i, site_j])
        maximum_selected_pair_error = max(
            maximum_selected_pair_error,
            abs(empirical_pga_rho - target_pga_rho),
            abs(empirical_sa_rho - target_sa_rho),
        )
        empirical_rows.extend(
            [
                {
                    "case_name": case_name,
                    "imt": "PGA",
                    "site_i": site_i,
                    "site_j": site_j,
                    "distance_km": float(distance_km[site_i, site_j]),
                    "target_correlation": target_pga_rho,
                    "empirical_correlation": empirical_pga_rho,
                    "absolute_error": abs(empirical_pga_rho - target_pga_rho),
                },
                {
                    "case_name": case_name,
                    "imt": "SA0P4",
                    "site_i": site_i,
                    "site_j": site_j,
                    "distance_km": float(distance_km[site_i, site_j]),
                    "target_correlation": target_sa_rho,
                    "empirical_correlation": empirical_sa_rho,
                    "absolute_error": abs(empirical_sa_rho - target_sa_rho),
                },
            ]
        )

    maximum_abs_site_mean = float(
        max(np.max(np.abs(pga_mean_by_site)), np.max(np.abs(sa_mean_by_site)))
    )
    maximum_abs_site_std_error = float(
        max(
            np.max(np.abs(pga_std_by_site - 1.0)),
            np.max(np.abs(sa_std_by_site - 1.0)),
        )
    )
    cross_imt_rho_error = float(
        abs(np.mean(pga_sa_rho_by_site) - EXPECTED_CROSS_IMT_RHO)
    )
    append_check(
        empirical_check_rows,
        f"{case_name.lower()}_empirical_site_means",
        maximum_abs_site_mean <= 0.08,
        f"Maximum absolute site mean: {maximum_abs_site_mean:.6f}",
    )
    append_check(
        empirical_check_rows,
        f"{case_name.lower()}_empirical_site_standard_deviations",
        maximum_abs_site_std_error <= 0.08,
        f"Maximum absolute site standard-deviation error: {maximum_abs_site_std_error:.6f}",
    )
    append_check(
        empirical_check_rows,
        f"{case_name.lower()}_empirical_selected_pair_correlations",
        maximum_selected_pair_error <= 0.06,
        f"Maximum selected-pair correlation error: {maximum_selected_pair_error:.6f}",
    )
    append_check(
        empirical_check_rows,
        f"{case_name.lower()}_empirical_cross_imt_correlation",
        cross_imt_rho_error <= 0.03,
        (
            f"Mean same-site rho: {np.mean(pga_sa_rho_by_site):.6f}; "
            f"target: {EXPECTED_CROSS_IMT_RHO:.6f}"
        ),
    )
    del epsilon_pga, epsilon_sa

empirical_pair_table = pd.DataFrame(empirical_rows)
empirical_validation = pd.DataFrame(empirical_check_rows)
empirical_pair_path = (
    METADATA_DIR / "notebook_8_empirical_pair_correlations.csv"
)
empirical_validation_path = METADATA_DIR / "notebook_8_empirical_validation.csv"
empirical_pair_table.to_csv(empirical_pair_path, index=False)
empirical_validation.to_csv(empirical_validation_path, index=False)
if not bool(empirical_validation.loc[
    empirical_validation["severity"].eq("critical"), "passed"
].all()):
    display(empirical_validation)
    raise RuntimeError("Notebook 08 empirical validation failed.")

write_json(
    METADATA_DIR / "notebook_8_empirical_summary.json",
    {
        "pipeline_version": PIPELINE_VERSION,
        "validation_seed": VALIDATION_SEED,
        "generator": "numpy.random.PCG64DXSM",
        "realizations": EMPIRICAL_REALIZATIONS,
        "selected_pairs": [list(pair) for pair in selected_pairs],
        "checks": int(len(empirical_validation)),
        "failures": int((~empirical_validation["passed"]).sum()),
        "maximum_pair_correlation_error": float(
            empirical_pair_table["absolute_error"].max()
        ),
    },
)

print("=" * 78)
print("NOTEBOOK 08 CELL 4: EMPIRICAL VALIDATION PASSED")
print("=" * 78)
print(f"Validation realizations:      {EMPIRICAL_REALIZATIONS:,}")
print(f"Selected site pairs:          {len(selected_pairs):,}")
print(f"Critical checks:              {len(empirical_validation):,}")
print(
    "Maximum pair-correlation error: "
    f"{empirical_pair_table['absolute_error'].max():.6f}"
)
display(empirical_validation)

In [ ]:
plot_distances = np.linspace(0.0, 10.0, 501)
pair_distances = distance_km[np.triu_indices(EXPECTED_SITE_COUNT, k=1)]
maximum_portfolio_distance = float(np.max(pair_distances))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot(
    plot_distances,
    aldea_correlation(plot_distances, imt="PGA"),
    label="Aldea et al. PGA",
    linewidth=2.2,
)
axes[0].plot(
    plot_distances,
    aldea_correlation(plot_distances, imt="SA0P4"),
    label="Aldea et al. SA(0.4 s)",
    linewidth=2.2,
)
axes[0].plot(
    plot_distances,
    goda_atkinson_correlation(plot_distances),
    label="Goda and Atkinson",
    linewidth=2.2,
)
axes[0].axvline(
    maximum_portfolio_distance,
    color="black",
    linestyle="--",
    linewidth=1.3,
    label="Maximum portfolio separation",
)
axes[0].set(
    xlabel="Site separation (km)",
    ylabel="Within-event correlation",
    xlim=(0.0, 10.0),
    ylim=(0.0, 1.02),
    title="Selected spatial-correlation models",
)
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False, fontsize=9)

axes[1].hist(pair_distances, bins=40, color="#3B82F6", alpha=0.85)
axes[1].axvline(
    float(np.median(pair_distances)),
    color="black",
    linestyle="--",
    linewidth=1.3,
    label=f"Median = {np.median(pair_distances):.2f} km",
)
axes[1].set(
    xlabel="Building-pair separation (km)",
    ylabel="Number of building pairs",
    title="Frozen Seaside portfolio geometry",
)
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(frameon=False, fontsize=9)
fig.tight_layout()

plot_path = PLOT_DIR / "spatial_correlation_models_and_pair_distances.png"
temporary_plot_path = plot_path.with_suffix(".tmp.png")
fig.savefig(temporary_plot_path, dpi=180, bbox_inches="tight")
temporary_plot_path.replace(plot_path)
plt.show()

print(f"Saved diagnostic figure: {project_relative_path(plot_path)}")

In [ ]:
validation_paths = [
    cell1_validation_path,
    analytic_validation_path,
    empirical_validation_path,
]
validation_tables = [pd.read_csv(path) for path in validation_paths]

portable_artifact_paths = [
    cell1_validation_path,
    METADATA_DIR / "notebook_8_cell_1_summary.json",
    geometry_path,
    matrix_diagnostics_path,
    case_summary_path,
    model_specification_path,
    analytic_validation_path,
    empirical_pair_path,
    empirical_validation_path,
    METADATA_DIR / "notebook_8_empirical_summary.json",
    factor_path,
    plot_path,
]
handoff_checks: list[dict[str, Any]] = []
portable_path_strings = [
    project_relative_path(path) for path in portable_artifact_paths
]
append_check(
    handoff_checks,
    "artifact_paths_are_repository_relative",
    all(
        not Path(path_string).is_absolute()
        and ":" not in Path(path_string).parts[0]
        for path_string in portable_path_strings
    ),
    f"Portable paths validated: {len(portable_path_strings)}.",
)
deterministic_json_paths = [
    METADATA_DIR / "notebook_8_cell_1_summary.json",
    model_specification_path,
    METADATA_DIR / "notebook_8_empirical_summary.json",
    geometry_path,
]
append_check(
    handoff_checks,
    "metadata_excludes_runtime_timestamps",
    all(
        "created_at_utc" not in load_json(path)
        for path in deterministic_json_paths
    ),
    "Runtime timestamps are excluded from hash-tracked JSON artifacts.",
)
validation_tables.append(pd.DataFrame(handoff_checks))
final_validation = pd.concat(validation_tables, ignore_index=True)
final_validation_path = METADATA_DIR / "notebook_8_final_validation.csv"
final_validation.to_csv(final_validation_path, index=False)
critical_mask = final_validation["severity"].eq("critical")
critical_failures = int((critical_mask & ~final_validation["passed"]).sum())
if critical_failures:
    display(final_validation.loc[critical_mask & ~final_validation["passed"]])
    raise RuntimeError(
        f"Notebook 08 final validation has {critical_failures} critical failures."
    )

artifact_paths = [
    cell1_validation_path,
    METADATA_DIR / "notebook_8_cell_1_summary.json",
    geometry_path,
    matrix_diagnostics_path,
    case_summary_path,
    model_specification_path,
    analytic_validation_path,
    empirical_pair_path,
    empirical_validation_path,
    METADATA_DIR / "notebook_8_empirical_summary.json",
    final_validation_path,
    factor_path,
    plot_path,
]
artifact_inventory = [
    {
        "path": project_relative_path(path),
        "sha256": sha256_file(path),
        "bytes": int(path.stat().st_size),
    }
    for path in artifact_paths
]

final_handoff = {
    "schema_version": "notebook8_spatial_correlation_handoff_v2",
    "pipeline_version": PIPELINE_VERSION,
    "notebook8_complete": True,
    "frozen_controls": {
        "phase1_release": "v1.0.0",
        "phase1_commit": "be93474ce2ab78d8002d49ae861adb641ae2741d",
        "site_order_canonical_crlf_sha256": site_order_crlf_sha256,
        "event_catalog_sha256": notebook4_handoff["annual_catalog"]["sha256"],
        "catalog_years": notebook4_handoff["annual_catalog"][
            "declared_duration_years"
        ],
        "occurrences": notebook4_handoff["annual_catalog"]["occurrences"],
        "cross_imt_rho": EXPECTED_CROSS_IMT_RHO,
        "random_generator": random_spec["generator"],
        "latent_transform": random_spec["latent_transform"],
    },
    "cases": list(SUPPORTED_CASES),
    "geometry": portfolio_geometry,
    "factor_artifact": {
        "path": project_relative_path(factor_path),
        "sha256": sha256_file(factor_path),
    },
    "model_specification": {
        "path": project_relative_path(model_specification_path),
        "sha256": sha256_file(model_specification_path),
    },
    "validation": {
        "path": project_relative_path(final_validation_path),
        "sha256": sha256_file(final_validation_path),
        "checks": int(len(final_validation)),
        "critical_checks": int(critical_mask.sum()),
        "critical_failures": critical_failures,
    },
    "artifact_inventory": artifact_inventory,
    "next_notebook": "09_generate_correlated_ground_motion_fields.ipynb",
    "next_task": (
        "Regenerate the two frozen site-level latent vectors for each of the "
        "10,630 catalog occurrences, apply the validated I0, C1, and C2 "
        "transforms, and rebuild PGA and SA0P4 without changing medians, "
        "between-event residuals, tau, or phi."
    ),
}
final_handoff_path = METADATA_DIR / "notebook_8_final_handoff.json"
write_json(final_handoff_path, final_handoff)

print("=" * 78)
print("NOTEBOOK 08 COMPLETE: SPATIAL-CORRELATION ENGINE VALIDATED")
print("=" * 78)
print(f"Dependence cases:             {len(SUPPORTED_CASES):,}")
print(f"Portfolio sites:              {EXPECTED_SITE_COUNT:,}")
print(f"Unique coordinates:           {EXPECTED_UNIQUE_COORDINATES:,}")
print(f"Validation checks:            {len(final_validation):,}")
print(f"Critical failures:            {critical_failures:,}")
print(f"Factor artifact:              {project_relative_path(factor_path)}")
print(f"Final handoff:                {project_relative_path(final_handoff_path)}")
print("Next: Notebook 09 full-catalog correlated ground-motion fields.")